# 🎯 SFDAO - Synthetic Finance Data Auditor & Optimizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/takurot/sfdao/blob/main/notebooks/sfdao_demo_en.ipynb)
[![PyPI version](https://badge.fury.io/py/sfdao.svg)](https://badge.fury.io/py/sfdao)

In this notebook, we demonstrate the key features of SFDAO.

## 📋 Contents

1. **Installation** - Install from PyPI
2. **Data Preparation** - Create sample data
3. **Synthetic Data Generation** - Simple generation feature
4. **Basic Audit** - Using `sfdao audit` command
5. **Python API** - Programmatic usage
6. **Score Integration** - Comprehensive evaluation
7. **Phase 2 Workflow** - Generation → Guardrails → Audit
8. **Visualization** - Distribution comparison

---
## 1. 📦 Installation

In [ ]:
# Install SFDAO
!pip install -q sfdao

# Check version
!sfdao --version

---
## 2. 📊 Data Preparation

We will prepare sample data for this demonstration. Here, we generate synthetic financial transaction data.

In [ ]:
import pandas as pd
import numpy as np

# Fix random seed
np.random.seed(42)

# Generate sample "Real Data"
n_samples = 1000

real_data = pd.DataFrame({
    'Time': np.sort(np.random.uniform(0, 172800, n_samples)),
    'V1': np.random.normal(0, 1, n_samples),
    'V2': np.random.normal(0, 1, n_samples),
    'V3': np.random.normal(0, 1, n_samples),
    'V4': np.random.normal(0, 1.5, n_samples),
    'V5': np.random.normal(0, 1, n_samples),
    'Amount': np.abs(np.random.lognormal(3, 1.5, n_samples)),
    'Class': np.random.choice([0, 1], n_samples, p=[0.95, 0.05])
})

# Save as CSV
real_data.to_csv('real_data.csv', index=False)
print(f"✅ Real data created: {len(real_data)} rows")
real_data.head()

---
## 3. 🔬 Synthetic Data Generation

We use the baseline generator included in SFDAO to create synthetic data.

In [ ]:
# Use the synthetic data generation script
!python -m sfdao.scripts.generate_test_synthetic_data \
    real_data.csv \
    synthetic_data.csv \
    --n-samples 1000 \
    --random-state 42

In [ ]:
# Check generated synthetic data
synthetic_data = pd.read_csv('synthetic_data.csv')
print(f"✅ Synthetic data: {len(synthetic_data)} rows")
synthetic_data.head()

---
## 4. 📋 Basic Audit (CLI)

Evaluate the quality of synthetic data using the `sfdao audit` command.

In [ ]:
# Output text report
!sfdao audit \
    --real real_data.csv \
    --synthetic synthetic_data.csv \
    --output report.txt

# Display report content
with open('report.txt', 'r') as f:
    print(f.read())

In [ ]:
# Output HTML report
!sfdao audit \
    --real real_data.csv \
    --synthetic synthetic_data.csv \
    --output report.html

print("✅ HTML report saved to report.html")

In [ ]:
# Display HTML report in Colab
from IPython.display import IFrame, display, HTML

with open('report.html', 'r') as f:
    html_content = f.read()

display(HTML(html_content))

---
## 5. 🐍 Python API Usage

You can use SFDAO features directly via Python API as well as CLI.

In [ ]:
from sfdao.ingestion.loader import load_csv
from sfdao.ingestion.type_detector import TypeDetector

# Load data
real_df = load_csv('real_data.csv')
synthetic_df = load_csv('synthetic_data.csv')

# Automatic Type Detection
detector = TypeDetector()
schema = detector.detect_all(real_df)

print("📊 Detected Column Types:")
for col, col_type in schema.items():
    print(f"  - {col}: {col_type.value}")

In [ ]:
from sfdao.evaluator.statistical import StatisticalEvaluator

# Run Statistical Evaluation
stat_evaluator = StatisticalEvaluator()
stat_result = stat_evaluator.evaluate(real_df, synthetic_df)

print("📈 Statistical Evaluation Results:")
print(f"  Overall Score: {stat_result.overall_score:.3f}")
print(f"  Mean JS Divergence: {stat_result.mean_js_divergence:.4f}")
print(f"  Mean KS Statistic: {stat_result.mean_ks_statistic:.4f}")

In [ ]:
from sfdao.evaluator.privacy import PrivacyEvaluator

# Run Privacy Evaluation
privacy_evaluator = PrivacyEvaluator()
privacy_result = privacy_evaluator.evaluate(real_df, synthetic_df)

print("🔒 Privacy Evaluation Results:")
print(f"  Overall Score: {privacy_result.overall_score:.3f}")
print(f"  Re-identification Risk: {privacy_result.reidentification_risk:.4f}")
print(f"  Mean DCR: {privacy_result.mean_dcr:.4f}")
print(f"  Min DCR: {privacy_result.min_dcr:.4f}")

In [ ]:
from sfdao.evaluator.finance_facts import FinancialFactsChecker

# Financial Stylized Facts
finance_checker = FinancialFactsChecker()
finance_result = finance_checker.evaluate(real_df, synthetic_df)

print("💰 Financial Facts Evaluation:")
print(f"  Overall Score: {finance_result.overall_score:.3f}")
print(f"  Fat Tail Preserved: {finance_result.fat_tail_preserved}")
print(f"  Volatility Clustering: {finance_result.volatility_clustering_preserved}")

---
## 6. ⚙️ Score Integration

Integrate multiple evaluation results to calculate the final score.

In [ ]:
from sfdao.evaluator.scorer import Scorer

# Integrated evaluation by Scorer
scorer = Scorer()
final_score = scorer.calculate_final_score(
    statistical_result=stat_result,
    privacy_result=privacy_result,
    finance_result=finance_result
)

print("\n" + "="*50)
print("🎯 FINAL EVALUATION SUMMARY")
print("="*50)
print(f"  Statistical Score:  {stat_result.overall_score:.3f}")
print(f"  Privacy Score:      {privacy_result.overall_score:.3f}")
print(f"  Finance Score:      {finance_result.overall_score:.3f}")
print(f"  ─────────────────────────────")
print(f"  📊 FINAL SCORE:     {final_score:.3f}")
print("="*50)

---
## 7. 🔄 Phase 2: Workflow Automation

The `sfdao run` command allows batch processing based on a configuration file.

In [ ]:
# Create config file for Phase 2
config_yaml = """
generator:
  type: baseline
  n_samples: 500
  random_state: 42

guard:
  rules:
    - column: Amount
      type: range
      min: 0
      max: 50000
    - column: Class
      type: enum
      values: [0, 1]

scenario:
  enabled: false

audit:
  statistical: true
  privacy: true
  finance: true

output:
  format: html
"""

with open('config.yaml', 'w') as f:
    f.write(config_yaml)

print("✅ Config file created: config.yaml")

In [ ]:
# Execute Phase 2 Workflow
!sfdao run \
    --real real_data.csv \
    --config config.yaml \
    --out-dir output_phase2

In [ ]:
# Check output files
import os

output_dir = 'output_phase2'
if os.path.exists(output_dir):
    print(f"📁 Output files in {output_dir}:")
    for f in os.listdir(output_dir):
        print(f"  - {f}")
else:
    print("⚠️ Output directory not found")

---
## 8. 📊 Visualization

Visualize and compare distributions of real and synthetic data.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

columns_to_plot = ['V1', 'V2', 'V3', 'V4', 'Amount', 'Class']

for ax, col in zip(axes.flatten(), columns_to_plot):
    ax.hist(real_df[col], bins=30, alpha=0.6, label='Real', density=True)
    ax.hist(synthetic_df[col], bins=30, alpha=0.6, label='Synthetic', density=True)
    ax.set_title(f'{col} Distribution')
    ax.legend()
    ax.set_xlabel(col)
    ax.set_ylabel('Density')

plt.tight_layout()
plt.suptitle('Real vs Synthetic Data Distribution', y=1.02, fontsize=14)
plt.savefig('distribution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Distribution comparison saved to distribution_comparison.png")

---
## 9. 🧹 Clean Up

In [ ]:
# List of generated files
import os

files = [
    'real_data.csv', 'synthetic_data.csv', 
    'report.txt', 'report.html', 
    'config.yaml', 'distribution_comparison.png'
]

print("📁 Generated files:")
for f in files:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"  ✅ {f} ({size:,} bytes)")

# Uncomment below to clean up files
# for f in files:
#     if os.path.exists(f):
#         os.remove(f)
# print("\n🧹 Files cleaned up")

---
## 📚 Next Steps

- **Documentation**: [GitHub Repository](https://github.com/takurot/sfdao)
- **PyPI**: `pip install sfdao`
- **Advanced Features**: `pip install sfdao[deep]` (CTGAN support)

### Key Features

| Feature | Description |
|---------|-------------|
| Statistical Evaluation | Distribution comparison via KS test & JS Divergence |
| Privacy Evaluation | Re-identification risk, Distance to Closest Record (DCR) |
| Financial Facts | Fat Tail detection, Volatility Clustering verification |
| Auto Type Detection | Automatic classification of Numeric, Categorical, Datetime, PII |
| Reporting | HTML/PDF/TXT formats |
| Guardrails | Rule-based constraints |
| Scenario Injection | Testing scale, shift, clip, outliers, etc. |

---

**Thank you for using SFDAO!** 🎉